# *Physarum polycephalum*

Preprocesamiento del experimento de *Physarum polycephalum* con el dataset de Harrington y Harrington (Zenodo,
DOI [10.5281/zenodo.22868368](https://doi.org/10.5281/zenodo.22868368), CC BY 4.0).

Son 24 cajas de Petri de 90 mm (experimentos 001, 002, 008 y 009, con 6 cajas cada uno) escaneadas desde abajo cada
5 min (001, 002) o 2 min (008, 009) durante 63–141 h: 51 036 imágenes RGB de ~1169×1276 px (0.085 mm/px). Cada caja es
un archivo ZIP sin compresión con un OME-Zarr v3 (`s0` = imágenes (T, 3, H, W) uint8, `acquisition/timestamp_utc_ns`,
`registration/dish_circle_xyr_px`, …). La avena dentro de la caja es el alimento del estado inicial.

1. **Descarga** de los 24 ZIP (133 GB) a `Dataset/Physarum/zarr/`.
2. **Muestreo**: una imagen cada N minutos, recortada a la caja → `Dataset/Physarum/cada<N>min/<caja>.npz`.
3. **Timelapse** de las 24 cajas → `Dataset/Physarum/timelapse_cada<N>min.mp4`.
4. **Limpieza**: círculo fijo, mediana temporal, radio útil y campo plano → `Dataset/Physarum/limpio_cada<N>min/<caja>.npz`.
5. **Estados**: borde, fondo, avena y plasmodio en 3 niveles → `CA/Physarum/estados_cada<N>min/<caja>.npy`;
   reducidos a 160 × 160 y sin parpadeo → `estados_cada<N>min_160/`.
6. **Video de los pasos** → `Dataset/Physarum/pasos_cada<N>min.mp4`.

In [ ]:
import hashlib
import re
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import requests
from PIL import Image

TRAINING = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "main.py").exists())
DATA = TRAINING / "Dataset" / "Physarum"
ZARR = DATA / "zarr"
RECORD = "https://zenodo.org/api/records/22868368"

## 1. Descarga

Baja los archivos del registro de Zenodo con `jobs` descargas a la vez (~10 MB/s en total, ~3 h 15 min). Un archivo a
medias se reanuda donde se quedó, así que si se corta basta volver a correr la celda. Cada ZIP se verifica contra
`SHA256SUMS` del dataset y, si coincide, se marca con `<archivo>.ok` para no volver a verificarlo.

In [ ]:
def bajar(url, destino, intentos=20):
    """Descarga url a destino, reanudando lo que ya exista."""
    for intento in range(intentos):
        hecho = destino.stat().st_size if destino.exists() else 0
        try:
            with requests.get(url, headers={"Range": f"bytes={hecho}-"} if hecho else {}, stream=True,
                              timeout=120) as r:
                if r.status_code == 416:                               # ya estaba completo
                    return
                if r.status_code == 429:
                    time.sleep(float(r.headers.get("Retry-After", 30)))
                    continue
                r.raise_for_status()
                with open(destino, "ab" if r.status_code == 206 else "wb") as f:
                    for bloque in r.iter_content(1 << 20):
                        f.write(bloque)
            return
        except requests.RequestException as e:
            print(f"  {destino.name}: {e!r}, reintento", flush=True)
            time.sleep(30)
    raise RuntimeError(f"no se pudo bajar {url}")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while bloque := f.read(1 << 24):
            h.update(bloque)
    return h.hexdigest()


def descargar(jobs=3):
    ZARR.mkdir(parents=True, exist_ok=True)
    archivos = [f["key"] for f in requests.get(RECORD, timeout=60).json()["files"]]
    for k in archivos:                                                 # documentos (README, SHA256SUMS, …)
        if not k.endswith(".ome.zarr.zip") and not (DATA / k).exists():
            bajar(f"{RECORD}/files/{k}/content", DATA / k)
    sumas = dict(reversed(l.split()) for l in (DATA / "SHA256SUMS").read_text().splitlines())

    def uno(k):
        ok = ZARR / f"{k}.ok"
        if ok.exists():
            return f"{k}: ya estaba"
        bajar(f"{RECORD}/files/{k}/content", ZARR / k)
        if sha256(ZARR / k) != sumas[k]:
            return f"{k}: SHA256 NO COINCIDE (borrarlo y volver a correr)"
        ok.touch()
        return f"{k}: OK"

    with ThreadPoolExecutor(jobs) as ex:
        for msg in ex.map(uno, sorted(k for k in archivos if k.endswith(".ome.zarr.zip"))):
            print(msg, flush=True)
    print(f"{len(list(ZARR.glob('*.ok')))}/24 archivos verificados")

In [ ]:
descargar()

## 2. Muestreo

Por cada caja:

1. Toma una imagen cada `every_min` minutos (por el timestamp real, no el nominal).
2. Recorta un cuadrado centrado en el círculo de la caja ajustado en esa imagen (`registration/dish_circle_xyr_px`; la
   cámara se mueve un poco entre imágenes), con el radio mediano de la serie × `margin`, y lo voltea en x (el escáner ve
   la caja desde abajo).
3. Reduce a `size` × `size` y guarda `Dataset/Physarum/cada<every_min>min/<caja>.npz` con `frames` (T, size, size, 3)
   uint8, `t_s` (s desde la primera imagen) e `idx` (índice original).

In [ ]:
def crop(frame: np.ndarray, x: float, y: float, r: int, size: int) -> np.ndarray:
    """frame (3, H, W) → cuadrado de lado 2r centrado en (x, y), relleno con 0 fuera, volteado en x."""
    img = frame.transpose(1, 2, 0)
    H, W = img.shape[:2]
    x0, y0 = int(round(x)) - r, int(round(y)) - r
    out = np.zeros((2 * r, 2 * r, 3), np.uint8)
    sx, sy = max(0, -x0), max(0, -y0)
    ex, ey = min(2 * r, W - x0), min(2 * r, H - y0)
    out[sy:ey, sx:ex] = img[y0 + sy:y0 + ey, x0 + sx:x0 + ex]
    return np.asarray(Image.fromarray(out[:, ::-1]).resize((size, size), Image.LANCZOS))


def muestrear_caja(nombre, every_min=30, size=320, margin=1.06, carpeta=None):
    import zarr

    carpeta = Path(carpeta or DATA / f"cada{every_min:g}min")
    path = carpeta / f"{nombre}.npz"
    if path.exists():
        return f"{nombre}: ya estaba"
    t0 = time.time()
    root = zarr.open_group(zarr.storage.ZipStore(ZARR / f"{nombre}.ome.zarr.zip", mode="r"), mode="r", zarr_format=3)
    s0 = root["s0"]
    ts = root["acquisition/timestamp_utc_ns"][:]
    circ = root["registration/dish_circle_xyr_px"][:]
    t_s = (ts - ts[0]) / 1e9
    # índice de la primera imagen en o después de cada múltiplo de every_min
    marcas = np.arange(0, t_s[-1] + 1, every_min * 60)
    idx = np.unique(np.clip(np.searchsorted(t_s, marcas), 0, len(t_s) - 1))
    r = int(round(np.median(circ[:, 2]) * margin))
    frames = np.stack([crop(s0[int(i)], circ[i, 0], circ[i, 1], r, size) for i in idx])
    carpeta.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp.npz")
    np.savez_compressed(tmp, frames=frames, t_s=t_s[idx], idx=idx)
    tmp.rename(path)
    return f"{nombre}: {len(idx)} imágenes en {time.time() - t0:.0f} s"


def muestrear(every_min=30, size=320, margin=1.06, cajas=None, workers=4):
    cajas = cajas or sorted(p.name[:-len(".ome.zarr.zip")] for p in ZARR.glob("*.ome.zarr.zip"))
    with ThreadPoolExecutor(workers) as ex:
        for msg in ex.map(lambda n: muestrear_caja(n, every_min, size, margin), cajas):
            print(msg, flush=True)

In [ ]:
muestrear(every_min=30, size=320)

## 3. Timelapse

Una fila por experimento (001, 002, 008, 009) y una columna por caja (dish0–dish5), con un solo reloj en horas desde la
primera imagen de cada experimento: en cada paso, cada caja muestra su última imagen con t ≤ reloj. Las cajas cuyo
experimento ya terminó se quedan en su última imagen, oscurecidas y con "fin". Escribe un MP4 (H.264 con libopenh264).

In [ ]:
HEADER, LABEL, LEFT = 44, 22, 120
FONT_PATHS = ["/usr/share/fonts/liberation-mono-fonts/LiberationMono-Regular.ttf",
              "/usr/share/fonts/dejavu-sans-fonts/DejaVuSans.ttf"]


def font(size):
    from PIL import ImageFont

    for p in FONT_PATHS:
        if Path(p).exists():
            return ImageFont.truetype(p, size)
    return ImageFont.load_default(size)


def timelapse(folder="cada30min", tile=240, fps=10):
    from PIL import ImageDraw

    series = {}
    for f in sorted((DATA / folder).glob("physarum-*.npz")):
        exp, dish = re.match(r"physarum-(\d+)-dish(\d)", f.stem).groups()
        d = np.load(f)
        tiles = np.stack([np.asarray(Image.fromarray(x).resize((tile, tile), Image.LANCZOS)) for x in d["frames"]])
        series[(exp, int(dish))] = (d["t_s"] / 3600, tiles)
    exps = sorted({e for e, _ in series})
    paso = float(np.median(np.diff(next(iter(series.values()))[0])))
    fin = max(t[-1] for t, _ in series.values())
    relojes = np.arange(0, fin + paso / 2, paso)

    W = LEFT + 6 * tile
    H = HEADER + len(exps) * (tile + LABEL)
    W, H = W + W % 2, H + H % 2
    out = DATA / f"timelapse_{folder}.mp4"
    ffmpeg = subprocess.Popen(
        ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{W}x{H}",
         "-r", str(fps), "-i", "-", "-c:v", "libopenh264", "-b:v", "6M", "-pix_fmt", "yuv420p", str(out)],
        stdin=subprocess.PIPE)
    f_titulo, f_txt = font(24), font(15)
    for reloj in relojes:
        img = Image.new("RGB", (W, H), (20, 20, 20))
        dr = ImageDraw.Draw(img)
        dr.text((12, 8), f"Physarum polycephalum   {reloj:6.1f} h", fill=(235, 235, 235), font=f_titulo)
        for i, e in enumerate(exps):
            y = HEADER + i * (tile + LABEL)
            dr.text((10, y + tile // 2 - 10), f"exp. {e}", fill=(235, 235, 235), font=f_txt)
            for j in range(6):
                if (e, j) not in series:
                    continue
                t, tiles = series[(e, j)]
                k = int(np.searchsorted(t, reloj + 1e-6) - 1)
                termino = reloj > t[-1] + paso / 2
                cuadro = tiles[max(k, 0)]
                if termino:
                    cuadro = (cuadro * 0.45).astype(np.uint8)
                x = LEFT + j * tile
                img.paste(Image.fromarray(cuadro), (x, y))
                etiqueta = f"dish{j}  " + ("fin" if termino else f"{t[max(k, 0)]:.1f} h")
                dr.text((x + 6, y + tile + 2), etiqueta, fill=(200, 200, 200), font=f_txt)
        ffmpeg.stdin.write(img.tobytes())
    ffmpeg.stdin.close()
    ffmpeg.wait()
    print(f"{out}: {len(relojes)} imágenes, {len(relojes) / fps:.1f} s, {len(series)} cajas")

In [ ]:
timelapse("cada30min")

## 4. Limpieza

El muestreo de la sección 2 sirve para ver el dataset, pero no para entrenar. Problemas medidos:

- **Temblor artificial.** La cámara no se mueve (la correlación de fase entre imágenes originales da < 0.25 px en toda
  la serie y el brillo fuera de la caja es constante, 31–36), pero el círculo que trae el dataset por imagen está
  cuantizado a 4 px y salta hasta 36 px en el centro y 56 px en el radio. Recortar con él mete saltos de hasta 15 px
  (a 320 px) entre imágenes consecutivas, que la NCA vería como movimiento.
- **Borde.** Pared de la caja y su reflejo (una media luna que se ve a través del fondo, sobre todo en las cajas de
  vidrio de 18 mm de 008/009), cinta en el borde (001/002) y plasmodio que sube por la pared o sale de la caja.
- **Color distinto entre cajas.** El fondo inicial va de ≈ 40 (008/009) a ≈ 85 (001/002) y hay viñeteo leve
  (≈ 6 niveles del centro al borde) y rayas del escáner.
- **Ruido** del sensor y cambios de un solo cuadro (≈ 1.3 niveles entre imágenes consecutivas).

Por cada caja, leyendo directamente del ZIP:

1. **Círculo fijo**: la mediana de `dish_circle_xyr_px` de toda la serie.
2. **Mediana temporal**: cada imagen muestreada (una cada `every_min` minutos) es la mediana por píxel de las imágenes
   originales dentro de ±`ventana_min`/2 (2–3 imágenes en 001/002, 5 en 008/009 con 10 min).
3. **Radio útil**: recorte fijo del cuadrado inscrito en `r_util`·R (0.90: deja fuera la pared y su reflejo), volteado
   en x (el escáner ve la caja desde abajo) y reducido a `size` × `size`; fuera del círculo = 0 (`dentro` = máscara).
4. **Campo plano**: se divide cada canal entre el fondo de la primera imagen y se multiplica por `fondo_ref` (60). Fondo
   = píxeles sin avena ni inóculo (luminancia < 1.4 × mediana, croma < mediana + 12), rellenado bajo la avena por
   convolución normalizada (σ = 1.5 px donde hay fondo, 12 px bajo la avena). Es uno solo por caja: los cambios de
   brillo en el tiempo son reales (crecimiento y rastro del plasmodio), no de iluminación.

Guarda `Dataset/Physarum/limpio_cada<every_min>min/<caja>.npz` con `frames` (T, size, size, 3) uint8, `t_s`, `idx`,
`n_prom` (imágenes por mediana), `dentro`, `circulo` (x, y, R en px originales), `r_util`, `plano` y `fondo_ref`.
`revisar_limpieza` compara el salto máximo entre imágenes consecutivas antes y después, y guarda un mosaico de las 24
cajas al inicio, a la mitad y al final.

In [ ]:
def abrir(nombre):
    import zarr

    return zarr.open_group(zarr.storage.ZipStore(ZARR / f"{nombre}.ome.zarr.zip", mode="r"), mode="r", zarr_format=3)


def recorte_fijo(img, x, y, half, size):
    """img (H, W, 3) → cuadrado de lado 2·half centrado en (x, y), relleno con 0, volteado en x, a size × size."""
    H, W = img.shape[:2]
    x0, y0 = int(round(x)) - half, int(round(y)) - half
    out = np.zeros((2 * half, 2 * half, 3), np.uint8)
    sx, sy = max(0, -x0), max(0, -y0)
    ex, ey = min(2 * half, W - x0), min(2 * half, H - y0)
    out[sy:ey, sx:ex] = img[y0 + sy:y0 + ey, x0 + sx:x0 + ex]
    return np.asarray(Image.fromarray(out[:, ::-1]).resize((size, size), Image.LANCZOS))


def mascara(size):
    """Círculo inscrito en la cuadrícula (el recorte ya es el radio útil)."""
    c = (size - 1) / 2
    yy, xx = np.mgrid[:size, :size]
    return np.hypot(yy - c, xx - c) <= size / 2


def campo_plano(frame, dentro, brillo=1.4, croma_extra=12, fino=1.5, grueso=12):
    """Fondo (size, size, 3) de la primera imagen y la máscara de los píxeles de fondo usados."""
    from scipy import ndimage as ndi

    f = frame.astype(np.float32)
    g = f.mean(-1)
    croma = f.max(-1) - f.min(-1)
    fondo = dentro & (g < brillo * np.median(g[dentro])) & (croma < np.median(croma[dentro]) + croma_extra)
    fondo = ndi.binary_erosion(fondo, iterations=2) & dentro
    m = fondo.astype(np.float32)

    def conv(x, s):                                                    # convolución normalizada
        return ndi.gaussian_filter(x * m, s) / np.maximum(ndi.gaussian_filter(m, s), 1e-6)

    w = np.clip(ndi.gaussian_filter(m, fino), 0, 1) ** 2               # 1 donde hay fondo, 0 bajo la avena
    plano = np.stack([w * conv(f[..., c], fino) + (1 - w) * conv(f[..., c], grueso) for c in range(3)], -1)
    return np.maximum(plano, 1).astype(np.float32), fondo


def limpiar_caja(nombre, every_min=30, ventana_min=10, size=320, r_util=0.90, fondo_ref=60.0, carpeta=None):
    carpeta = Path(carpeta or DATA / f"limpio_cada{every_min:g}min")
    path = carpeta / f"{nombre}.npz"
    if path.exists():
        return f"{nombre}: ya estaba"
    t0 = time.time()
    root = abrir(nombre)
    s0 = root["s0"]
    ts = root["acquisition/timestamp_utc_ns"][:]
    x, y, R = np.median(root["registration/dish_circle_xyr_px"][:], axis=0)    # 1. círculo fijo
    t_s = (ts - ts[0]) / 1e9
    marcas = np.arange(0, t_s[-1] + 1, every_min * 60)
    idx = np.unique(np.clip(np.searchsorted(t_s, marcas), 0, len(t_s) - 1))
    half = int(round(R * r_util))
    dentro = mascara(size)

    frames = np.empty((len(idx), size, size, 3), np.uint8)
    n_prom = np.empty(len(idx), np.int16)
    for k, i in enumerate(idx):
        vecinas = np.flatnonzero(np.abs(t_s - t_s[i]) <= ventana_min * 30)      # 2. mediana en ±ventana/2
        med = np.round(np.median(s0[int(vecinas[0]):int(vecinas[-1]) + 1], axis=0)).astype(np.uint8)
        frames[k] = recorte_fijo(med.transpose(1, 2, 0), x, y, half, size)     # 3. radio útil
        n_prom[k] = len(vecinas)

    plano, _ = campo_plano(frames[0], dentro)                                  # 4. campo plano
    limpio = np.clip(frames.astype(np.float32) / plano * fondo_ref, 0, 255).round().astype(np.uint8)
    limpio[:, ~dentro] = 0

    carpeta.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp.npz")
    np.savez_compressed(tmp, frames=limpio, t_s=t_s[idx], idx=idx, n_prom=n_prom, dentro=dentro,
                        circulo=np.array([x, y, R]), r_util=r_util, plano=plano, fondo_ref=fondo_ref)
    tmp.rename(path)
    return f"{nombre}: {len(idx)} imágenes, R = {R:.0f} px, en {time.time() - t0:.0f} s"


def limpiar(every_min=30, ventana_min=10, size=320, r_util=0.90, fondo_ref=60.0, cajas=None, workers=4):
    cajas = cajas or sorted(p.name[:-len(".ome.zarr.zip")] for p in ZARR.glob("*.ome.zarr.zip"))
    with ThreadPoolExecutor(workers) as ex:
        for msg in ex.map(lambda n: limpiar_caja(n, every_min, ventana_min, size, r_util, fondo_ref), cajas):
            print(msg, flush=True)


def salto_maximo(frames):
    """Mayor desplazamiento (px) entre imágenes consecutivas, por correlación de fase."""
    from skimage.registration import phase_cross_correlation

    g = frames.astype(np.float32).mean(-1)
    return max(np.abs(phase_cross_correlation(g[i - 1], g[i], upsample_factor=2)[0]).max() for i in range(1, len(g)))


def revisar_limpieza(every_min=30, tile=200):
    limpio = sorted((DATA / f"limpio_cada{every_min:g}min").glob("physarum-*.npz"))
    print(f"{'caja':24} salto antes  después  fondo inicial (RGB)")
    filas = {0: [], 0.5: [], 1: []}
    for f in limpio:
        d = np.load(f)
        antes = DATA / f"cada{every_min:g}min" / f.name
        s_antes = f"{salto_maximo(np.load(antes)['frames']):11.1f}" if antes.exists() else f"{'—':>11}"
        fondo = np.median(d["frames"][0][d["dentro"]], axis=0)
        print(f"{f.stem:24} {s_antes}  {salto_maximo(d['frames']):7.1f}  {fondo}")
        for k in filas:
            filas[k].append(np.asarray(Image.fromarray(d["frames"][round(k * (len(d["frames"]) - 1))])
                                       .resize((tile, tile), Image.LANCZOS)))
    for k, cuadros in filas.items():
        mosaico = np.concatenate([np.concatenate(cuadros[i:i + 6], 1) for i in range(0, len(cuadros), 6)], 0)
        out = DATA / f"limpio_cada{every_min:g}min_{['inicio', 'mitad', 'final'][int(k * 2)]}.png"
        Image.fromarray(mosaico).save(out)
        print(out)

In [ ]:
limpiar(every_min=30, size=320)
revisar_limpieza(30)

## 5. Estados

Seis estados por célula (una célula = un píxel de la imagen limpia, 320 × 320, ≈ 0.24 mm):

| estado | nombre | regla |
|---:|---|---|
| 0 | borde | fuera del radio útil (`dentro` = falso); no cambia |
| 1 | fondo | todo lo demás |
| 2 | avena | máscara fija de la primera imagen (abajo) y sin plasmodio encima |
| 3 | plasmodio tenue | 9 ≤ b* < 16 |
| 4 | plasmodio medio | 16 ≤ b* < 26 |
| 5 | plasmodio intenso | b* ≥ 26 |

Se trabaja en CIELAB (L = luminosidad, a* = rojo–verde, b* = amarillo–azul, C = croma = √(a*² + b*²)), con mediana
3×3 por canal para quitar puntos sueltos. Tras el campo plano el fondo es gris (a* = b* = 0; en la primera imagen el
99.9 % del fondo tiene b* < 5–14, por eso el umbral de 9).

- **Plasmodio**: es amarillo y, cuanto más se mueve (frentes de avance, venas gruesas con flujo), más brillante y
  saturado se ve; b* sube con él (correlación 0.74 con L en los píxeles de plasmodio). Los niveles se cortan en b* = 9,
  16 y 26: el nivel tenue cubre las redes finas y el velo de 001/002, el intenso los frentes activos.
- **Avena**: no se mueve, así que se detecta una sola vez en la primera imagen: componentes con L > 38 y 12 < C < 38
  (el inóculo amarillo tiene C ≈ 43–53), sin los de área < 20 px ni los que son más de 8 unidades de L más oscuros que
  la mediana de los componentes de la caja (el inóculo café de 001 y el reflejo del borde). Se rellenan los huecos y
  se ensancha 2 px para absorber el borde difuso de cada hojuela, que si no sale como un anillo de plasmodio. En cada
  imagen, un píxel de la máscara es avena salvo que su b* haya subido más de 10 respecto de la primera imagen
  (plasmodio encima; entonces toma el nivel de su b*).

Escribe `CA/Physarum/estados_cada<every_min>min/<caja>.npy` (T, 320, 320) int8 (una evolución por caja: las cajas
duran distinto, `main.py` acepta varios `--data`), `evo.json` (reglas, umbrales y proporción de cada estado por caja) y
la paleta `GUI/colors/physarum6.csv`.

In [ ]:
import json

ESTADOS = ["borde", "fondo", "avena", "plasmodio tenue", "plasmodio medio", "plasmodio intenso"]
PALETA = ["#000000", "#2A2A30", "#E6CFA8", "#6E6A12", "#D2AE00", "#FFF46A"]
BORDE, FONDO, AVENA = 0, 1, 2
UMBRALES_B = (9, 16, 26)


def lab_suave(frame):
    """(H, W, 3) uint8 → [L, a*, b*] con mediana 3×3 por canal."""
    from scipy import ndimage as ndi
    from skimage.color import rgb2lab

    lab = rgb2lab(frame.astype(np.float32) / 255)
    return [ndi.median_filter(lab[..., c], 3) for c in range(3)]


def mascara_avena(frame0, dentro, L_min=38, C_min=12, C_max=38, oscura=8, area_min=20, crecer=2):
    """Avena en la primera imagen → (máscara, b* de la primera imagen)."""
    from scipy import ndimage as ndi

    L, a, b = lab_suave(frame0)
    C = np.hypot(a, b)
    av = dentro & (L > L_min) & (C > C_min) & (C < C_max)
    av = ndi.binary_fill_holes(ndi.binary_opening(av, iterations=1))
    lb, n = ndi.label(av)
    ids = np.arange(1, n + 1)
    grandes = ndi.sum(av, lb, ids) >= area_min
    L_media = ndi.mean(L, lb, ids)
    L_ref = np.median(L_media[grandes])
    av = np.isin(lb, ids[grandes & (L_media >= L_ref - oscura)])
    return ndi.binary_dilation(av, iterations=crecer) & dentro, b


def estados_frames(frames, dentro, umbrales=UMBRALES_B, delta_avena=10):
    """frames (T, H, W, 3) limpios → (T, H, W) int8: borde, fondo, avena y len(umbrales) niveles de plasmodio."""
    avena, b0 = mascara_avena(frames[0], dentro)
    S = np.empty(frames.shape[:3], np.int8)
    for k, x in enumerate(frames):
        _, _, b = lab_suave(x)
        nivel = np.digitize(b, umbrales)                           # 0 = sin plasmodio, 1–3 = tenue, medio, intenso
        solo_avena = avena & (b - b0 <= delta_avena)
        s = np.where(nivel > 0, AVENA + nivel, FONDO).astype(np.int8)
        s[solo_avena] = AVENA
        s[~dentro] = BORDE
        S[k] = s
    return S


def colorear(S):
    rgb = np.array([[int(h[i:i + 2], 16) for i in (1, 3, 5)] for h in PALETA], np.uint8)
    return rgb[S]


def nombres_y_paleta(umbrales):
    """Nombres y colores de los estados para len(umbrales) niveles de plasmodio (3 = los de la tabla)."""
    if tuple(umbrales) == tuple(UMBRALES_B):
        return ESTADOS, PALETA
    n = len(umbrales)
    a, b = np.array([0x6E, 0x6A, 0x12]), np.array([0xFF, 0xF4, 0x6A])          # de oliva a amarillo claro
    colores = ["#%02X%02X%02X" % tuple(np.round(a + (b - a) * k / max(n - 1, 1)).astype(int)) for k in range(n)]
    return ESTADOS[:3] + [f"plasmodio {k + 1}" for k in range(n)], PALETA[:3] + colores


def estados(every_min=30, umbrales=UMBRALES_B, sufijo="", workers=6):
    """sufijo distingue otros umbrales (p. ej. "_8estados" con 5 niveles de plasmodio)."""
    nombres, paleta = nombres_y_paleta(umbrales)
    carpeta = TRAINING / "CA" / "Physarum" / f"estados_cada{every_min:g}min{sufijo}"
    carpeta.mkdir(parents=True, exist_ok=True)
    limpio = sorted((DATA / f"limpio_cada{every_min:g}min").glob("physarum-*.npz"))

    def uno(f):
        d = np.load(f)
        S = estados_frames(d["frames"], d["dentro"], umbrales)
        np.save(carpeta / f"{f.stem}.npy", S)
        n = d["dentro"].sum()
        prop = lambda s: {e: round(float((s == i).sum() / n), 4) for i, e in enumerate(nombres) if i > 0}
        return {"caja": f.stem, "pasos": len(S), "inicio": prop(S[0]), "final": prop(S[-1])}

    with ThreadPoolExecutor(workers) as ex:
        cajas = list(ex.map(uno, limpio))
    (carpeta / "evo.json").write_text(json.dumps({
        "simulation": "physarum", "every_min": every_min, "size": 320, "estados": nombres, "paleta": paleta,
        "umbrales_b": list(umbrales), "avena": {"L_min": 38, "C_min": 12, "C_max": 38, "oscura": 8, "area_min": 20,
                                            "crecer": 2, "delta_b": 10},
        "pipeline": ["muestrear", "limpiar", "estados"], "cajas": cajas}, indent=1, ensure_ascii=False))
    (TRAINING.parent / "GUI" / "colors" / f"physarum{len(nombres)}.csv").write_text("\n".join(paleta) + "\n")
    print(f"{'caja':24} {'pasos':>5}  " + "  ".join(f"{e.split()[-1]:>9}" for e in nombres[1:]) + "   (final, fracción de `dentro`)")
    for c in cajas:
        print(f"{c['caja']:24} {c['pasos']:5}  " + "  ".join(f"{c['final'][e]:9.3f}" for e in nombres[1:]))
    print(carpeta)

In [ ]:
estados(every_min=30)

### 5.1 Reducción y filtro temporal

La primera NCA de prueba (`NCA/Physarum/prueba/`) aprendió la identidad: el 28 % de los cambios de estado se deshace
en el paso siguiente (células cerca de los umbrales de color que van y vienen), así que casi un tercio de lo que debía
aprender era parpadeo de la segmentación. `estados_reducidos` hace, sobre los estados de 320 × 320:

1. **Reducción** por `factor` (2 → 160 × 160, ≈ 0.48 mm por célula): cada bloque de factor × factor toma su estado más
   frecuente; en un empate gana el estado mayor (así una vena de 2 píxeles de ancho sobrevive y el borde solo queda
   donde el bloque está casi todo fuera).
2. **Filtro temporal**: una célula solo cambia a un estado nuevo si ese estado se mantiene `persistencia` pasos
   seguidos en los datos; si no, conserva el anterior. F[0] = S[0] y F[t] = S[t] si S[t] = S[t+1] = … =
   S[t+persistencia−1], si no F[t−1]. La última imagen conserva el estado filtrado anterior cuando no se puede
   comprobar.

Escribe `CA/Physarum/estados_cada<every_min>min_<160>/<caja>.npy` y su `evo.json` (proporción de cambios que se
revierten antes y después). Con `persistencia` distinta de 2 la carpeta termina en `_p<persistencia>`; `sufijo` lee
los estados de otros umbrales (`estados(umbrales=…, sufijo=…)`).

In [ ]:
def reducir_estados(S, factor):
    """(T, H, W) → (T, H/f, W/f): estado más frecuente de cada bloque, empates al estado mayor."""
    T, H, W = S.shape
    B = S.reshape(T, H // factor, factor, W // factor, factor).transpose(0, 1, 3, 2, 4).reshape(T, H // factor,
                                                                                                W // factor, -1)
    n = int(S.max()) + 1
    cuenta = np.stack([(B == e).sum(-1) for e in range(n)], -1)                            # (T, h, w, estados)
    return (n - 1 - cuenta[..., ::-1].argmax(-1)).astype(np.int8)                          # empate → estado mayor


def filtro_temporal(S, persistencia=2):
    F = S.copy()
    for t in range(1, len(S)):
        firme = np.ones(S.shape[1:], bool)
        for k in range(1, persistencia):
            if t + k >= len(S):                                        # no se puede comprobar: no cambia
                firme[:] = False
                break
            firme &= S[t + k] == S[t]
        F[t] = np.where(firme, S[t], F[t - 1])
    return F


def revierten(S):
    """Fracción de los cambios de estado (dentro) que se deshacen en el paso siguiente."""
    d = S[0] != BORDE
    a, b, c = S[:-2][:, d], S[1:-1][:, d], S[2:][:, d]
    cambia = a != b
    return float((cambia & (c == a)).sum() / max(cambia.sum(), 1))


def estados_reducidos(every_min=30, factor=2, persistencia=2, sufijo=""):
    """Lee estados_cada<N>min<sufijo>/ y escribe estados_cada<N>min<sufijo>_<lado>[_p<persistencia> si no es 2]/."""
    origen = TRAINING / "CA" / "Physarum" / f"estados_cada{every_min:g}min{sufijo}"
    lado = 320 // factor
    carpeta = TRAINING / "CA" / "Physarum" / (f"estados_cada{every_min:g}min{sufijo}_{lado}"
                                              + ("" if persistencia == 2 else f"_p{persistencia}"))
    carpeta.mkdir(parents=True, exist_ok=True)
    cajas = []
    for f in sorted(origen.glob("physarum-*.npy")):
        R = reducir_estados(np.load(f), factor)
        F = filtro_temporal(R, persistencia)
        np.save(carpeta / f.name, F)
        d = F[0] != BORDE
        cajas.append({"caja": f.stem, "revierten_antes": round(revierten(R), 4), "revierten_despues": round(revierten(F), 4),
                      "cambian_antes": round(float((R[1:] != R[:-1])[:, d].mean()), 4),
                      "cambian_despues": round(float((F[1:] != F[:-1])[:, d].mean()), 4)})
    (carpeta / "evo.json").write_text(json.dumps({
        "simulation": "physarum", "every_min": every_min, "size": lado, "factor": factor, "persistencia": persistencia,
        "estados": json.loads((origen / "evo.json").read_text())["estados"], "origen": origen.name,
        "pipeline": ["muestrear", "limpiar", "estados", "estados_reducidos"], "cajas": cajas}, indent=1,
        ensure_ascii=False))
    media = lambda k: np.mean([c[k] for c in cajas])
    print(f"{carpeta}: {len(cajas)} cajas {lado}×{lado}; cambios por paso {media('cambian_antes'):.3f} → "
          f"{media('cambian_despues'):.3f}, se revierten {media('revierten_antes'):.3f} → {media('revierten_despues'):.3f}")

In [ ]:
estados_reducidos(every_min=30, factor=2, persistencia=2)
# variantes del barrido nocturno (NCA/Physarum/barrido/)
estados(every_min=30, umbrales=(9, 13, 17, 22, 28), sufijo="_8estados")
estados_reducidos(every_min=30, factor=2, persistencia=2, sufijo="_8estados")
estados_reducidos(every_min=30, factor=2, persistencia=1)
estados_reducidos(every_min=30, factor=2, persistencia=3)
estados_reducidos(every_min=30, factor=4, persistencia=2)

## 6. Video de los pasos

`video_pasos` rehace la limpieza de algunas cajas guardando cada paso intermedio y escribe un MP4 con una fila por caja
y una columna por paso: 0) muestreo de la sección 2 (círculo de cada imagen, tiembla), 1) círculo fijo, 2) mediana
temporal, 3) radio útil, 4) campo plano (= `limpio_cada<N>min`), 5) estados con la paleta de `physarum6.csv`. Las
columnas 0–2 usan el recorte del muestreo (1.06 R); 3–5, el radio útil. El reloj es común (horas desde la primera
imagen); una caja cuyo experimento ya terminó se queda en su última imagen, oscurecida. Por default, una caja por
experimento, de las que más temblaban.

In [ ]:
PASOS = ["0. Muestreo", "1. Círculo fijo", "2. Mediana temporal", "3. Radio útil", "4. Campo plano",
         "5. Estados"]


def pasos_caja(nombre, every_min=30, ventana_min=10, size=320, margin=1.06, r_util=0.90, fondo_ref=60.0):
    """Los 6 pasos de una caja: (t_s, [6 arreglos (T, size, size, 3) uint8])."""
    root = abrir(nombre)
    s0 = root["s0"]
    ts = root["acquisition/timestamp_utc_ns"][:]
    circ = root["registration/dish_circle_xyr_px"][:]
    x, y, R = np.median(circ, axis=0)
    t_s = (ts - ts[0]) / 1e9
    marcas = np.arange(0, t_s[-1] + 1, every_min * 60)
    idx = np.unique(np.clip(np.searchsorted(t_s, marcas), 0, len(t_s) - 1))
    r_muestreo, half = int(round(R * margin)), int(round(R * r_util))
    dentro = mascara(size)

    pasos = [np.empty((len(idx), size, size, 3), np.uint8) for _ in range(4)]
    for k, i in enumerate(idx):
        img = s0[int(i)]
        pasos[0][k] = crop(img, circ[i, 0], circ[i, 1], r_muestreo, size)               # círculo de la imagen
        pasos[1][k] = recorte_fijo(img.transpose(1, 2, 0), x, y, r_muestreo, size)      # círculo fijo
        vecinas = np.flatnonzero(np.abs(t_s - t_s[i]) <= ventana_min * 30)
        med = np.round(np.median(s0[int(vecinas[0]):int(vecinas[-1]) + 1], axis=0)).astype(np.uint8)
        med = med.transpose(1, 2, 0)
        pasos[2][k] = recorte_fijo(med, x, y, r_muestreo, size)                         # mediana temporal
        pasos[3][k] = recorte_fijo(med, x, y, half, size) * dentro[..., None]           # radio útil
    plano, _ = campo_plano(pasos[3][0], dentro)
    limpio = np.clip(pasos[3].astype(np.float32) / plano * fondo_ref, 0, 255).round().astype(np.uint8)
    limpio[:, ~dentro] = 0                                                              # campo plano
    return t_s[idx], pasos + [limpio, colorear(estados_frames(limpio, dentro))]           # estados


def video_pasos(cajas=("physarum-001-dish1-r0c1", "physarum-002-dish0-r0c0", "physarum-008-dish2-r1c0",
                       "physarum-009-dish0-r0c0"), every_min=30, tile=240, fps=10, workers=4):
    from PIL import ImageDraw

    with ThreadPoolExecutor(workers) as ex:
        series = list(ex.map(lambda n: pasos_caja(n, every_min), cajas))
    series = [(t / 3600, [np.stack([np.asarray(Image.fromarray(x).resize((tile, tile), Image.LANCZOS)) for x in p])
                          for p in pasos]) for t, pasos in series]
    paso = float(np.median(np.diff(series[0][0])))
    relojes = np.arange(0, max(t[-1] for t, _ in series) + paso / 2, paso)

    sep = 6
    W = LEFT + len(PASOS) * (tile + sep)
    H = HEADER + LABEL + len(cajas) * (tile + sep) + 30                     # + franja de la leyenda
    W, H = W + W % 2, H + H % 2
    out = DATA / f"pasos_cada{every_min:g}min.mp4"
    ffmpeg = subprocess.Popen(
        ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{W}x{H}",
         "-r", str(fps), "-i", "-", "-c:v", "libopenh264", "-b:v", "8M", "-pix_fmt", "yuv420p", str(out)],
        stdin=subprocess.PIPE)
    f_titulo, f_txt = font(24), font(15)
    for reloj in relojes:
        img = Image.new("RGB", (W, H), (20, 20, 20))
        dr = ImageDraw.Draw(img)
        dr.text((12, 8), f"Physarum polycephalum: limpieza y estados   {reloj:6.1f} h", fill=(235, 235, 235), font=f_titulo)
        x_ley, y_ley = LEFT, H - 26
        for i, e in enumerate(ESTADOS):                                         # leyenda de colores
            dr.rectangle([x_ley, y_ley + 3, x_ley + 12, y_ley + 15], fill=PALETA[i], outline=(120, 120, 120))
            dr.text((x_ley + 16, y_ley), f"{i} {e}", fill=(220, 220, 220), font=f_txt)
            x_ley += int(dr.textlength(f" {i} {e}  ", font=f_txt)) + 14
        for j, nombre_paso in enumerate(PASOS):
            dr.text((LEFT + j * (tile + sep) + 6, HEADER), nombre_paso, fill=(235, 235, 235), font=f_txt)
        for i, (nombre, (t, pasos)) in enumerate(zip(cajas, series)):
            y = HEADER + LABEL + i * (tile + sep)
            exp, dish = re.match(r"physarum-(\d+)-dish(\d)", nombre).groups()
            k = max(int(np.searchsorted(t, reloj + 1e-6) - 1), 0)
            termino = reloj > t[-1] + paso / 2
            dr.text((10, y + tile // 2 - 20), f"exp. {exp}\ndish{dish}\n" + ("fin" if termino else f"{t[k]:.1f} h"),
                    fill=(220, 220, 220), font=f_txt)
            for j, p in enumerate(pasos):
                cuadro = p[k] if not termino else (p[k] * 0.45).astype(np.uint8)
                img.paste(Image.fromarray(cuadro), (LEFT + j * (tile + sep), y))
        ffmpeg.stdin.write(img.tobytes())
    ffmpeg.stdin.close()
    ffmpeg.wait()
    print(f"{out}: {len(relojes)} imágenes, {len(relojes) / fps:.1f} s, {len(cajas)} cajas × {len(PASOS)} pasos")

In [ ]:
video_pasos()